# Geografía social · SUBDERE — División Político Administrativa (límites comunales)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Observatory/datopia-notebooks/blob/main/notebooks/geografia-social/subdere/demo.ipynb)
[![Licencia](https://img.shields.io/badge/licencia-MIT-blue.svg)](../../../LICENSE)
[![Dataset](https://img.shields.io/badge/dataset-SUBDERE%20DPA-green.svg)]()
[![Actualización](https://img.shields.io/badge/actualización-estático-brightgreen.svg)]()
[![Python](https://img.shields.io/badge/python-3.10%2B-blue.svg)]()

---

## Descripción

Acceso a la **División Político Administrativa (DPA) de Chile** a través del **Datopia Lakehouse**:
los polígonos de las comunas con sus códigos (CUT) y nombres oficiales de región, provincia y
comuna, en formato **GeoParquet**.

La fuente `subdere` (`categoria=geografia-social`) tiene dos tablas:

| Tabla (`tipo=`) | Versión | Qué contiene |
|---|---|---|
| `division-territorial` | `v2` | Una fila por comuna con su polígono. Partición `snapshot_date=`: la edición base (**2018-09-06**, DPA 2023) y fotos históricas reconstruidas desde las leyes (desde **2000-05-08**). |
| `division-territorial-cambios` | `v1` | Un registro por unidad cambiada por cada ley o decreto (creaciones, traslados, cambios de nombre), con fechas y enlace a BCN LeyChile. |

Este notebook compara dos fotos de la DPA: la de **2000-05-08** (13 regiones) y la base de
**2018-09-06** (16 regiones), y usa la tabla de cambios para explicar cada diferencia.

El acceso es directo vía S3 con credenciales temporales, sin descarga de archivos.

### Contenido

1. Configuración y autenticación
2. Exploración: manifiesto, fotos disponibles y esquema
3. Evolución de la DPA en las fotos (solo atributos, sin leer geometría)
4. Las dos fotos: 2000-05-08 y 2018-09-06
5. Mapas de regiones en ambas fechas
6. Qué cambió: puente de códigos 2000 → 2018 con la tabla de cambios
7. Mapa de los cambios
8. Consulta espacial: ¿en qué comuna cae un punto en cada fecha?
9. Superficie: cifra oficial y área de la geometría

### Requisitos

Cuenta en el Datopia Lakehouse · Python 3.10+ · DuckDB 1.4 o superior (se instala automáticamente)

In [ ]:
# @title Instalación de dependencias
import importlib, subprocess, sys

for paquete in ["requests", "pandas", "matplotlib", "geopandas", "shapely"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

# Los archivos usan GeoParquet 2.0 (tipo GEOMETRY nativo de Parquet): DuckDB los lee desde la
# versión 1.4. Con una versión anterior y la extensión spatial cargada, la lectura falla.
from importlib.metadata import version, PackageNotFoundError

try:
    _version_duckdb = tuple(int(p) for p in version("duckdb").split(".")[:2])
except PackageNotFoundError:
    _version_duckdb = (0, 0)
if _version_duckdb < (1, 4):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "duckdb>=1.4"])

import json, os, getpass, pathlib
import requests, duckdb
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

print(f"DuckDB {duckdb.__version__} · GeoPandas {gpd.__version__}")
print("Dependencias listas.")

In [ ]:
# @title Configuración de credenciales
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")

URL_API = "https://ee98cnfz7e.execute-api.us-west-2.amazonaws.com/prod"

if EN_COLAB:
    EMAIL = input("Email: ").strip()
    PASSWORD = getpass.getpass("Contraseña: ")
else:
    ruta_cfg = pathlib.Path("../../../config.json")
    if not ruta_cfg.exists():
        raise FileNotFoundError(
            f"Archivo de configuración no encontrado en {ruta_cfg.resolve()}.\n"
            "Copia config.example.json → config.json y completa tus credenciales."
        )
    cfg = json.loads(ruta_cfg.read_text())
    EMAIL = cfg.get("test_user", {}).get("email") or input("Email: ").strip()
    PASSWORD = cfg.get("test_user", {}).get("password") or getpass.getpass(
        "Contraseña: "
    )

print(f"API    : {URL_API}")
usuario, _, dominio = EMAIL.partition("@")
print(f"Usuario: {usuario[:3]}***@{dominio}")   # se enmascara para no dejar el email en las salidas

---
## 1. Autenticación y conexión a S3

In [ ]:
# Iniciar sesión y obtener credenciales temporales S3
resp_login = requests.post(
    f"{URL_API}/auth/login",
    json={"email": EMAIL, "password": PASSWORD},
    timeout=30,
)
resp_login.raise_for_status()
token = resp_login.json()["id_token"]

resp_s3 = requests.post(
    f"{URL_API}/auth/session/s3",
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
resp_s3.raise_for_status()
creds = resp_s3.json()

print("Sesión iniciada")
print(f"  Bucket : {creds['bucket']}")
print(f"  Región : {creds['region']}")
print(f"  Expira : {creds['expires_at']}")

In [ ]:
# Configurar DuckDB con credenciales S3 y la extensión espacial
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute("INSTALL spatial; LOAD spatial")
con.execute(f"""
    CREATE OR REPLACE SECRET s3 (
        TYPE          S3,
        KEY_ID        '{creds["access_key_id"]}',
        SECRET        '{creds["secret_access_key"]}',
        SESSION_TOKEN '{creds["session_token"]}',
        REGION        '{creds["region"]}'
    )
""")
# Cachear metadatos (footers, listados) entre consultas: el notebook lee los mismos archivos varias veces
con.execute("SET enable_http_metadata_cache=true")
con.execute("SET parquet_metadata_cache=true")

BASE = f"s3://{creds['bucket']}/categoria=geografia-social/pais=cl/fuente=subdere"
BASE_DPA = f"{BASE}/tipo=division-territorial/version=v2"
BASE_CAMBIOS = f"{BASE}/tipo=division-territorial-cambios/version=v1"
print("DuckDB conectado a S3.")

---
## 2. Exploración del dataset

Cada foto de la DPA es una partición `snapshot_date=AAAA-MM-DD/` con un archivo
`part-<id de ejecución>-00000.parquet`. El archivo `manifest.json` de la versión lista las partes
vigentes: leer desde ahí (y no con un glob de todo el prefijo) asegura usar siempre la publicación
actual, aunque se esté re-ejecutando el pipeline.

In [ ]:
def leer_json_s3(ruta):
    """Lee un JSON de S3 a un dict (read_text no interpreta el contenido)."""
    return json.loads(con.execute(f"SELECT content FROM read_text('{ruta}')").fetchone()[0])


manifiesto = leer_json_s3(f"{BASE_DPA}/manifest.json")
print(f"Formato : {manifiesto.get('schema_version')}")
print(f"Filas   : {manifiesto['row_count']:,} en {len(manifiesto['parts'])} fotos")
for fuente_doc in manifiesto.get("sources", []):
    print(f"Fuente  : {fuente_doc.get('file')} · {fuente_doc.get('url')}")

fotos = pd.DataFrame(manifiesto["parts"])[["snapshot_date", "row_count", "bytes", "file"]]
fotos["MB"] = (fotos["bytes"] / 1e6).round(1)
fotos = fotos.sort_values("snapshot_date").reset_index(drop=True)

# Ruta S3 de cada foto, para leer exactamente los archivos del manifiesto
RUTA_FOTO = {f: f"{BASE_DPA}/{archivo}" for f, archivo in zip(fotos["snapshot_date"], fotos["file"])}
fotos[["snapshot_date", "row_count", "MB"]]

Cada archivo pesa del orden de 90 MB, casi todo en la columna `geometry` (polígonos de alta
resolución). Las consultas que solo leen atributos (`cut_*`, `nom_*`, `superficie`, `bbox_*`) no
descargan la geometría: Parquet es columnar.

In [ ]:
# Metadatos del catálogo (dataset.json). Puede no estar publicado aún para esta versión.
try:
    meta = con.execute(f"SELECT * FROM read_json('{BASE_DPA}/dataset.json')").df().iloc[0].to_dict()
    print(str(meta.get("description"))[:600])
    print()
    print(f"Formato     : {meta.get('format')}")
    print(f"Particiones : {meta.get('partitioned_by') or meta.get('partition_keys')}")
    print(f"Licencia    : {meta.get('license')}")
    columnas_catalogo = pd.DataFrame([dict(c) for c in meta["columns"]])[["name", "type", "comment"]]
    display(columnas_catalogo)
except Exception as error:
    print(f"dataset.json no disponible ({type(error).__name__}); el esquema se lee del Parquet abajo.")

In [ ]:
# Esquema del archivo (lee solo el footer del Parquet)
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{RUTA_FOTO["2018-09-06"]}', hive_partitioning=false)
""").df()[["column_name", "column_type"]]

| Columna | Descripción |
|---|---|
| `cut_reg`, `cut_prov`, `cut_com` | Código Único Territorial (2, 3 y 5 dígitos, texto con ceros a la izquierda). |
| `nom_reg`, `nom_prov`, `nom_com` | Nombres oficiales (Decreto 1.115 de 2018), con tildes. |
| `abr_reg` | Abreviatura de la región (`AYP`, `RM`, …). |
| `superficie` | km², cifra oficial de la fuente. Nula en Antártica y en comunas sin geometría reconstruible. |
| `geometry` | Polígono o multipolígono, EPSG:4326 (GeoParquet 2.0). Nula en Antártica Chilena (`12202`). |
| `bbox_xmin` … `bbox_ymax` | Caja envolvente de la geometría: permite filtrar espacialmente sin leer `geometry`. |
| `area_m2`, `perimeter_m` | Área y perímetro calculados sobre la geometría publicada, en metros. |

Los códigos son **texto**: `'08101'` y `8101` no son lo mismo al unir con otras tablas.

---
## 3. Evolución de la DPA en las fotos

Una sola consulta sobre los archivos de todas las fotos, leyendo solo columnas de códigos.

In [ ]:
lista_fotos = "[" + ", ".join(f"'{r}'" for r in RUTA_FOTO.values()) + "]"

evolucion = con.execute(f"""
    SELECT snapshot_date,
           COUNT(DISTINCT cut_reg)  AS regiones,
           COUNT(DISTINCT cut_prov) AS provincias,
           COUNT(*)                 AS comunas
    FROM read_parquet({lista_fotos}, hive_partitioning=true)
    GROUP BY snapshot_date
    ORDER BY snapshot_date
""").df()
evolucion

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.step(evolucion["snapshot_date"], evolucion["comunas"], where="post", color="#1565C0", linewidth=2)
ax.scatter(evolucion["snapshot_date"], evolucion["comunas"], color="#1565C0", zorder=3)
cambia_regiones = evolucion["regiones"].ne(evolucion["regiones"].shift())
for _, fila in evolucion[cambia_regiones].iterrows():
    ax.annotate(f"{fila['regiones']} reg.", (fila["snapshot_date"], fila["comunas"]),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_title("Comunas en cada foto de la DPA (y regiones cuando cambian)", fontweight="bold")
ax.set_ylabel("Comunas")
ax.yaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Las dos fotos: 2000-05-08 y 2018-09-06

Se cargan ambas fotos como `GeoDataFrame` para dibujarlas. DuckDB lee la geometría GeoParquet 2.0,
la **simplifica** (`ST_Simplify`, tolerancia ~300 m) y la entrega como WKB, que GeoPandas convierte
a polígonos: los polígonos originales tienen millones de vértices y no hacen falta para un mapa
nacional. Las consultas exactas (sección 8) usan la geometría original, directamente en DuckDB.
Es la celda más pesada del notebook (~180 MB desde S3).

In [ ]:
FECHA_ANTES = "2000-05-08"
FECHA_DESPUES = "2018-09-06"

COLUMNAS = "cut_reg, nom_reg, abr_reg, cut_prov, nom_prov, cut_com, nom_com, superficie, area_m2"
TOLERANCIA_GRADOS = 0.003   # ~300 m: suficiente para mapas regionales


def cargar_foto(fecha):
    df = con.execute(f"""
        SELECT {COLUMNAS},
               ST_AsWKB(ST_MakeValid(ST_Simplify(geometry, {TOLERANCIA_GRADOS}))) AS wkb
        FROM read_parquet('{RUTA_FOTO[fecha]}', hive_partitioning=false)
        ORDER BY cut_com
    """).df()
    wkb = [bytes(b) if isinstance(b, (bytes, bytearray, memoryview)) else None for b in df.pop("wkb")]
    geometria = gpd.GeoSeries.from_wkb(wkb, crs="EPSG:4326")   # Antártica: sin geometría
    return gpd.GeoDataFrame(df, geometry=geometria)


dpa_antes = cargar_foto(FECHA_ANTES)
dpa_despues = cargar_foto(FECHA_DESPUES)

for fecha, gdf in [(FECHA_ANTES, dpa_antes), (FECHA_DESPUES, dpa_despues)]:
    print(f"{fecha}: {len(gdf)} comunas · {gdf['cut_prov'].nunique()} provincias · "
          f"{gdf['cut_reg'].nunique()} regiones · sin geometría: {', '.join(gdf.loc[gdf.geometry.isna(), 'nom_com'])}")
dpa_despues.drop(columns="geometry").head()

In [ ]:
# Regiones en cada foto: código, nombre y número de comunas
regiones = (
    pd.concat([dpa_antes.assign(foto=FECHA_ANTES), dpa_despues.assign(foto=FECHA_DESPUES)])
    .groupby(["cut_reg", "foto"])
    .agg(nom_reg=("nom_reg", "first"), comunas=("cut_com", "count"))
    .unstack("foto")
)
regiones.columns = [f"{a}_{b}" for a, b in regiones.columns]
regiones

---
## 5. Mapas de regiones en ambas fechas

El color depende del **código** de región, así que una región que no cambió de código tiene el
mismo color en ambos mapas. Las regiones se arman uniendo sus comunas (`dissolve`). El mapa se
limita al Chile continental (las islas oceánicas quedan fuera del encuadre).

In [ ]:
from shapely.geometry import MultiPolygon, Polygon


def sin_huecos_chicos(geom, area_min=0.01):
    """Quita los huecos chicos que deja la simplificación entre comunas vecinas al unirlas."""
    poligonos = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
    return MultiPolygon([
        Polygon(p.exterior, [r for r in p.interiors if Polygon(r).area >= area_min])
        for p in poligonos
    ])


def regiones_de(gdf):
    """Regiones como unión de sus comunas."""
    reg = gdf.dropna(subset=["geometry"]).dissolve(by="cut_reg", as_index=False, aggfunc={"nom_reg": "first"})
    reg["geometry"] = reg.geometry.map(sin_huecos_chicos)
    return reg


regiones_antes = regiones_de(dpa_antes)
regiones_despues = regiones_de(dpa_despues)

codigos_region = sorted(set(regiones_antes["cut_reg"]) | set(regiones_despues["cut_reg"]))
paleta = plt.get_cmap("tab20")
COLOR_REGION = {codigo: paleta(i % 20) for i, codigo in enumerate(codigos_region)}

ENCUADRE = dict(xlim=(-76, -66), ylim=(-56.2, -17.3))   # Chile continental

fig, axes = plt.subplots(1, 2, figsize=(11, 15))
for ax, fecha, reg in [(axes[0], FECHA_ANTES, regiones_antes), (axes[1], FECHA_DESPUES, regiones_despues)]:
    reg.plot(ax=ax, color=[COLOR_REGION[c] for c in reg["cut_reg"]], edgecolor="black", linewidth=0.5)
    ax.set(**ENCUADRE)
    ax.set_title(f"{fecha}\n{len(reg)} regiones", fontweight="bold")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")

leyenda = [
    Patch(facecolor=COLOR_REGION[c], edgecolor="black",
          label=f"{c} {regiones_despues.set_index('cut_reg')['nom_reg'].get(c, regiones_antes.set_index('cut_reg')['nom_reg'].get(c))}")
    for c in codigos_region
]
fig.legend(handles=leyenda, loc="lower center", ncol=3, fontsize=8, frameon=False)
fig.suptitle("División regional de Chile", fontweight="bold", fontsize=14)
plt.tight_layout(rect=(0, 0.07, 1, 0.98))
plt.show()

---
## 6. Qué cambió: puente de códigos 2000 → 2018

Comparar las fotos por `cut_com` muestra muchas diferencias, pero la mayoría no son comunas nuevas:
cuando se crea una región, sus comunas **cambian de código** (por ejemplo, Valdivia pasa de la
región 10 a la 14). La tabla `division-territorial-cambios` registra cada cambio con su ley.

In [ ]:
codigos_antes = set(dpa_antes["cut_com"])
codigos_despues = set(dpa_despues["cut_com"])
print(f"Códigos de comuna solo en {FECHA_ANTES}  : {len(codigos_antes - codigos_despues)}")
print(f"Códigos de comuna solo en {FECHA_DESPUES}: {len(codigos_despues - codigos_antes)}")
print(f"Códigos en ambas fechas          : {len(codigos_antes & codigos_despues)}")

In [ ]:
# Tabla de cambios (tabla chica, plana): leer las partes listadas en su manifiesto
manifiesto_cambios = leer_json_s3(f"{BASE_CAMBIOS}/manifest.json")
lista_cambios = "[" + ", ".join(f"'{BASE_CAMBIOS}/{p['file']}'" for p in manifiesto_cambios["parts"]) + "]"
cambios = con.execute(f"SELECT * FROM read_parquet({lista_cambios}, hive_partitioning=false)").df()
print(f"Filas: {len(cambios)} · normas: {cambios['norma_id'].nunique()}")

# Tipos de registro: 'define' son las filas de los decretos que fijan la tabla completa (no son cambios)
cambios.groupby(["nivel", "tipo_cambio"]).size().unstack(fill_value=0)

In [ ]:
# Normas vigentes entre las dos fotos (la foto de una fecha ya incluye los cambios vigentes ese día)
entre_fotos = cambios[
    (cambios["tipo_cambio"] != "define")
    & (pd.to_datetime(cambios["fecha_vigencia"]) > pd.Timestamp(FECHA_ANTES))
    & (pd.to_datetime(cambios["fecha_vigencia"]) <= pd.Timestamp(FECHA_DESPUES))
]

normas = (
    entre_fotos.groupby(["fecha_vigencia", "norma_id", "norma_url"])
    .agg(cambios=("nivel", "size"),
         detalle=("tipo_cambio", lambda s: ", ".join(f"{k}: {v}" for k, v in s.value_counts().items())))
    .reset_index()
    .sort_values("fecha_vigencia")
)
normas

In [ ]:
# Comunas creadas entre las dos fotos y la comuna de la que se separaron.
# La tabla de cambios escribe los nombres tal como la ley (con sus erratas, p. ej. "Hulapén");
# el nombre oficial sale de la foto 2018.
creadas = entre_fotos[(entre_fotos["nivel"] == "comuna") & (entre_fotos["tipo_cambio"] == "creacion")]
(
    creadas[["fecha_vigencia", "norma_id", "cut_anterior", "cut_nuevo", "nombre_nuevo"]]
    .rename(columns={"nombre_nuevo": "nombre_en_la_ley"})
    .merge(dpa_despues[["cut_com", "nom_com"]], left_on="cut_nuevo", right_on="cut_com")
    .merge(dpa_antes[["cut_com", "nom_com"]].rename(columns={"cut_com": "cut_anterior", "nom_com": "separada_de"}), on="cut_anterior")
    .drop(columns="cut_com")
)

Con los traslados (cambios de código, en orden de vigencia) se arma un **puente**: para cada
comuna de 2000, su código en 2018. Si la tabla de cambios está completa, los códigos de 2000
trasladados más las comunas creadas tienen que dar exactamente los códigos de 2018.

In [ ]:
traslados = (
    entre_fotos[(entre_fotos["nivel"] == "comuna") & (entre_fotos["tipo_cambio"] == "traslado")]
    .sort_values("fecha_vigencia")
)

puente = {codigo: codigo for codigo in codigos_antes}          # código 2000 → código actual
# Cada norma se aplica de una vez (no fila por fila), por si una misma ley intercambia códigos
for _, norma in traslados.groupby(["fecha_vigencia", "norma_id"], sort=True):
    mapeo = dict(zip(norma["cut_anterior"], norma["cut_nuevo"]))
    puente = {origen: mapeo.get(actual, actual) for origen, actual in puente.items()}

puente_df = pd.DataFrame({"cut_com_2000": list(puente), "cut_com_2018": list(puente.values())})
reconstruido = set(puente_df["cut_com_2018"]) | set(creadas["cut_nuevo"])

print(f"Comunas que cambiaron de código: {(puente_df['cut_com_2000'] != puente_df['cut_com_2018']).sum()}")
print(f"Comunas creadas                : {len(creadas)}")
print(f"¿Puente + creaciones = códigos {FECHA_DESPUES}? {reconstruido == codigos_despues}")
if reconstruido != codigos_despues:
    print("  sobran :", sorted(reconstruido - codigos_despues))
    print("  faltan :", sorted(codigos_despues - reconstruido))

In [ ]:
# Comunas que cambiaron de código, con nombre y región antes y después
nombres_antes = dpa_antes.set_index("cut_com")[["nom_com", "nom_reg"]]
nombres_despues = dpa_despues.set_index("cut_com")[["nom_com", "nom_reg"]]

recodificadas = (
    puente_df[puente_df["cut_com_2000"] != puente_df["cut_com_2018"]]
    .join(nombres_antes, on="cut_com_2000")
    .join(nombres_despues, on="cut_com_2018", rsuffix="_2018")
    .rename(columns={"nom_com": "nom_com_2000", "nom_reg": "nom_reg_2000"})
    .sort_values("cut_com_2018")
    .reset_index(drop=True)
)
print("Comunas que cambiaron de región o provincia (por región de destino):")
print(recodificadas.groupby(["nom_reg_2000", "nom_reg_2018"]).size().to_string())
recodificadas.head(10)

In [ ]:
# Comunas con el mismo origen pero distinto nombre (cambios de nombre)
renombradas = recodificadas[recodificadas["nom_com_2000"] != recodificadas["nom_com_2018"]]
mismas = puente_df[puente_df["cut_com_2000"] == puente_df["cut_com_2018"]]["cut_com_2000"]
renombradas_mismo_codigo = (
    nombres_antes.loc[mismas, ["nom_com"]]
    .join(nombres_despues[["nom_com"]], rsuffix="_2018")
    .query("nom_com != nom_com_2018")
)
print(f"Cambios de nombre: {len(renombradas) + len(renombradas_mismo_codigo)}")
pd.concat([
    renombradas[["cut_com_2000", "nom_com_2000", "cut_com_2018", "nom_com_2018"]],
    renombradas_mismo_codigo.reset_index().rename(columns={"cut_com": "cut_com_2000", "nom_com": "nom_com_2000"})
        .assign(cut_com_2018=lambda d: d["cut_com_2000"]),
])

---
## 7. Mapa de los cambios

Comunas de 2018 coloreadas según lo que les pasó desde 2000, con los **límites regionales de 2000**
en línea negra gruesa. Tres acercamientos donde se crearon regiones: Arica y Parinacota (2007),
Los Ríos (2007) y Ñuble (2018).

In [ ]:
creadas_set = set(creadas["cut_nuevo"])
recodificadas_set = set(recodificadas["cut_com_2018"])

CATEGORIAS = {
    "creada": ("Comuna creada", "#D32F2F"),
    "recodificada": ("Cambió de región o provincia", "#FFA000"),
    "sin_cambio": ("Sin cambio de código", "#CFD8DC"),
}

mapa_cambios = dpa_despues.dropna(subset=["geometry"]).copy()
mapa_cambios["estado"] = mapa_cambios["cut_com"].map(
    lambda c: "creada" if c in creadas_set else "recodificada" if c in recodificadas_set else "sin_cambio"
)

ACERCAMIENTOS = [
    ("Arica y Parinacota (2007)", (-70.6, -68.3), (-21.7, -17.4)),
    ("Los Ríos (2007)",           (-73.9, -71.4), (-41.0, -39.2)),
    ("Ñuble (2018)",              (-73.3, -70.9), (-37.9, -35.9)),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, (titulo, xlim, ylim) in zip(axes, ACERCAMIENTOS):
    for estado, (_, color) in CATEGORIAS.items():
        sub = mapa_cambios[mapa_cambios["estado"] == estado]
        if len(sub):
            sub.plot(ax=ax, color=color, edgecolor="white", linewidth=0.4)
    regiones_antes.boundary.plot(ax=ax, color="black", linewidth=1.6)
    vista = mapa_cambios.cx[xlim[0]:xlim[1], ylim[0]:ylim[1]]
    for _, fila in vista[vista["estado"] != "sin_cambio"].iterrows():
        punto = fila.geometry.representative_point()
        if xlim[0] < punto.x < xlim[1] and ylim[0] < punto.y < ylim[1]:
            ax.annotate(fila["nom_com"], (punto.x, punto.y), fontsize=6, ha="center")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.set_aspect("equal")

fig.legend(
    handles=[Patch(facecolor=c, label=n) for n, c in CATEGORIAS.values()]
    + [Patch(facecolor="none", edgecolor="black", linewidth=1.6, label=f"Límite regional {FECHA_ANTES}")],
    loc="lower center", ncol=4, frameon=False,
)
fig.suptitle(f"Cambios en la DPA entre {FECHA_ANTES} y {FECHA_DESPUES}", fontweight="bold", fontsize=14)
plt.tight_layout(rect=(0, 0.05, 1, 0.95))
plt.show()

---
## 8. Consulta espacial: ¿en qué comuna cae un punto en cada fecha?

El patrón recomendado en DuckDB tiene dos pasos: primero filtrar por las columnas `bbox_*` (solo
estadísticas y números, sin decodificar polígonos) y después aplicar el predicado exacto
`ST_Contains`. Así se responde, por ejemplo, a qué comuna pertenecía una dirección en un año dado.

Los predicados espaciales (`ST_*`) funcionan en DuckDB o GDAL, **no en Athena**.

In [ ]:
LUGARES = pd.DataFrame(
    [
        ("Centro de Alto Hospicio", -70.1050, -20.2680),
        ("Plaza de Armas de Valdivia", -73.2454, -39.8142),
        ("Plaza de Armas de Chillán", -72.1034, -36.6066),
        ("Plaza de Armas de Arica", -70.3126, -18.4783),
        ("Plaza de Armas de Santiago", -70.6506, -33.4378),
    ],
    columns=["lugar", "lon", "lat"],
)
con.register("lugares", LUGARES)

ubicaciones = con.execute(f"""
    SELECT l.lugar, d.snapshot_date, d.cut_com, d.nom_com, d.nom_reg
    FROM lugares l
    JOIN read_parquet(['{RUTA_FOTO[FECHA_ANTES]}', '{RUTA_FOTO[FECHA_DESPUES]}'], hive_partitioning=true) d
      ON  d.bbox_xmin <= l.lon AND d.bbox_xmax >= l.lon              -- (1) caja envolvente
      AND d.bbox_ymin <= l.lat AND d.bbox_ymax >= l.lat
      AND ST_Contains(d.geometry, ST_Point(l.lon, l.lat))           -- (2) predicado exacto
    ORDER BY l.lugar, d.snapshot_date
""").df()

ubicaciones.pivot(index="lugar", columns="snapshot_date", values=["nom_com", "nom_reg"])

In [ ]:
# Lo mismo para una sola coordenada y cualquier foto disponible
def comuna_en(lon, lat, fecha=FECHA_DESPUES):
    """Comuna que contiene el punto (lon, lat) en la foto `fecha` (una de RUTA_FOTO)."""
    return con.execute(f"""
        SELECT cut_com, nom_com, nom_prov, nom_reg
        FROM read_parquet('{RUTA_FOTO[fecha]}', hive_partitioning=false)
        WHERE bbox_xmin <= ? AND bbox_xmax >= ? AND bbox_ymin <= ? AND bbox_ymax >= ?
          AND ST_Contains(geometry, ST_Point(?, ?))
    """, [lon, lon, lat, lat, lon, lat]).df()


print("Fotos disponibles:", ", ".join(RUTA_FOTO))

# Un punto del Alto Biobío: era parte de Santa Bárbara hasta la ley 19.959 (vigente el 2004-07-21)
pd.concat([comuna_en(-71.35, -37.95, f).assign(foto=f) for f in ["2004-04-22", "2004-07-21"]])

---
## 9. Superficie: cifra oficial y área de la geometría

Hay dos números de área por comuna:

- `superficie` (km²): la cifra oficial de la fuente, calculada por SUBDERE/IGM reproyectando cada
  comuna a su huso UTM. Es la que corresponde citar.
- `area_m2`: calculada sobre la geometría publicada. Sirve para comparar entre fotos o con
  geometrías propias, pero no coincide exactamente con la cifra oficial.

Las coordenadas están en grados (EPSG:4326): para medir en metros hay que reproyectar. En DuckDB,
escribir siempre `ST_Transform(geometry, 'EPSG:4326', 'EPSG:32719', always_xy := true)`; sin
`always_xy` los ejes quedan invertidos.

In [ ]:
# Superficie oficial por región en ambas fechas (la región que pierde comunas pierde superficie)
superficie_region = (
    pd.concat([dpa_antes.assign(foto=FECHA_ANTES), dpa_despues.assign(foto=FECHA_DESPUES)])
    .groupby(["cut_reg", "foto"])
    .agg(nom_reg=("nom_reg", "first"), superficie_km2=("superficie", "sum"))
    .unstack("foto")
)
superficie_region.columns = [f"{a}_{b}" for a, b in superficie_region.columns]
superficie_region = superficie_region.drop(columns=f"nom_reg_{FECHA_ANTES}").round(0)
superficie_region

In [ ]:
# Oficial vs. geometría, y un cálculo propio reproyectando a UTM 19S en DuckDB (tres comunas)
con.execute(f"""
    SELECT cut_com, nom_com,
           superficie                                         AS superficie_oficial_km2,
           ROUND(area_m2 / 1e6, 2)                            AS area_geometria_km2,
           ROUND(ST_Area(ST_Transform(geometry, 'EPSG:4326', 'EPSG:32719', always_xy := true)) / 1e6, 2)
                                                              AS area_utm19s_km2
    FROM read_parquet('{RUTA_FOTO[FECHA_DESPUES]}', hive_partitioning=false)
    WHERE cut_com IN ('13101', '01107', '05201')
    ORDER BY cut_com
""").df()

---
### Notas

- **Antártica Chilena** (`12202`) está en todas las fotos pero sin geometría ni superficie.
- Las fotos históricas se reconstruyen desde la base 2018-09-06 revirtiendo los cambios de la tabla
  `division-territorial-cambios`; cada fila de esa tabla trae el enlace a la norma en BCN LeyChile
  (`norma_url`).
- Los nombres son los oficiales del Decreto 1.115 de 2018 en todas las fotos, para que una misma
  comuna se llame igual en cualquier fecha.
- Lectores: DuckDB ≥1.4, GeoPandas ≥1.1, pyarrow ≥20 y GDAL ≥3.11 leen GeoParquet 2.0. Polars
  todavía no.
- Licencia de los datos: CC BY-NC-SA 4.0.

*Datos provistos por [Data Observatory](https://dataobservatory.net) · Fuente: SUBDERE / IGM / DIFROL / INE (DPA 2023, geoportal.cl) y BCN LeyChile*